In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# Digit Recognizer - MNIST (Kaggle)
# Author: ChatGPT (GPT-5)
# Compatible with Kaggle Notebook Environment

# =========================================
# 1️⃣ Import Libraries
# =========================================
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt

# =========================================
# 2️⃣ Load Data
# =========================================
train = pd.read_csv("/kaggle/input/digit-recognizer/train.csv")
test = pd.read_csv("/kaggle/input/digit-recognizer/test.csv")

print("Train shape:", train.shape)
print("Test shape:", test.shape)

# =========================================
# 3️⃣ Split features and labels
# =========================================
X = train.drop("label", axis=1).values
y = train["label"].values

# Normalize pixel values (0–255 → 0–1)
X = X / 255.0
test = test / 255.0

# Reshape for CNN (images are 28x28 pixels)
X = X.reshape(-1, 28, 28, 1)
test = test.values.reshape(-1, 28, 28, 1)

# =========================================
# 4️⃣ Train-Test Split for Validation
# =========================================
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.1, random_state=42)

# =========================================
# 5️⃣ Define CNN Model
# =========================================
model = keras.Sequential([
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(28,28,1)),
    layers.Conv2D(32, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),
    layers.Dropout(0.25),

    layers.Conv2D(64, (3,3), activation='relu'),
    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),
    layers.Dropout(0.25),

    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(10, activation='softmax')
])

# =========================================
# 6️⃣ Compile Model
# =========================================
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

model.summary()

# =========================================
# 7️⃣ Train Model
# =========================================
history = model.fit(
    X_train, y_train,
    epochs=15,
    batch_size=64,
    validation_data=(X_val, y_val),
    verbose=2
)

# =========================================
# 8️⃣ Evaluate on Validation Set
# =========================================
val_loss, val_acc = model.evaluate(X_val, y_val)
print("Validation accuracy:", round(val_acc*100, 2), "%")

# =========================================
# 9️⃣ Plot Training Curves
# =========================================
plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
plt.plot(history.history['accuracy'], label='train acc')
plt.plot(history.history['val_accuracy'], label='val acc')
plt.legend()
plt.title('Accuracy')

plt.subplot(1,2,2)
plt.plot(history.history['loss'], label='train loss')
plt.plot(history.history['val_loss'], label='val loss')
plt.legend()
plt.title('Loss')
plt.show()

# =========================================
# 🔟 Predict on Test Data
# =========================================
predictions = np.argmax(model.predict(test), axis=1)

# Create submission file
submission = pd.DataFrame({
    "ImageId": np.arange(1, len(predictions)+1),
    "Label": predictions
})
submission.to_csv("submission.csv", index=False)

print("✅ submission.csv file created successfully!")
